# Invoke FM (Amazon Nova 2 Lite) for Generating Text (Email Responses) Using Bedrock API

**Lab:** Generate personalized customer email responses using Amazon Nova 2 Lite via Amazon Bedrock and Boto3.

**Model Used:** `us.amazon.nova-2-lite-v1:0` (Active — replaces deprecated Nova Lite v1)

**Region:** `us-west-2`

---

## Scenario
You are an AI/ML Engineer at K21Technologies. Your company receives a significant amount of customer feedback including both positive and negative responses. Your task is to generate personalized and empathetic responses to customers who have expressed dissatisfaction, using a Large Language Model (LLM) to automate the process.

## What You Will Learn
- Use Amazon Bedrock with Boto3 for text generation
- Apply zero-shot prompting to generate context-aware outputs
- Generate human-like email responses using Amazon Nova 2 Lite
- Extract and clean the generated text for practical use
- Use both standard and streaming response modes

---
## Cell 1 — Install Required Libraries

Run this cell to install and import the required packages.

> **Note:** You won't see any output after running this cell — that is normal. If no error appears, the libraries were successfully installed. ✅

In [1]:
# Install required packages
import sys
!{sys.executable} -m pip install boto3 --quiet

# Import required libraries
import boto3
import json

# Create the Amazon Bedrock Runtime client
# Nova 2 Lite is available via cross-region inference in us-west-2
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-west-2'
)

print("✅ Libraries imported and Bedrock client created successfully.")

✅ Libraries imported and Bedrock client created successfully.


---
## Cell 2 — Define the Customer Feedback (Input)

This cell defines the customer's negative feedback email that we want to respond to.

> **Note:** There is no output for this cell.

In [2]:
# Customer's negative feedback email
customer_email = """
Subject: Extremely Disappointed with Your Service

Dear Support Team,

I am writing to express my extreme disappointment with the service I received from your company.
I have been a loyal customer for over three years, but my recent experience has been nothing short of terrible.

I placed an order two weeks ago and still haven't received it. When I contacted your support team,
I was kept on hold for over an hour and then told that my order was lost in transit.
No one proactively informed me about this issue.

I expect an immediate resolution and a full explanation of how this happened.
If this is not resolved promptly, I will have no choice but to take my business elsewhere.

Regards,
John Smith
"""

print("✅ Customer feedback defined successfully.")

✅ Customer feedback defined successfully.


---
## Cell 3 — Build the Prompt and Request Body

This cell constructs the zero-shot prompt and the request body in the format required by Amazon Nova 2 Lite.

> **Note:** There is no output for this cell.

**Model ID:** `us.amazon.nova-2-lite-v1:0`  
The `us.` prefix is required for cross-region inference profiles in us-east-1.

In [3]:
# Model ID for Amazon Nova 2 Lite
# The 'us.' prefix enables cross-region inference (required for Nova 2 models)
MODEL_ID = "us.amazon.nova-2-lite-v1:0"

# Zero-shot prompt: describe the task in plain language
prompt = f"""You are a professional customer service representative.
A customer has sent the following negative feedback email:

{customer_email}

Write a professional, empathetic, and solution-oriented response email to this customer.
The response should:
- Acknowledge the customer's frustration
- Apologize sincerely for the inconvenience
- Provide a clear plan to resolve the issue
- Offer compensation if appropriate
- End on a positive note

Write only the email response, starting with 'Subject:'"""

# Build the request body in Amazon Nova 2 Lite message format
request_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 500,
        "temperature": 0.7,
        "topP": 0.9
    }
}

print("✅ Prompt and request body prepared successfully.")
print(f"   Model ID: {MODEL_ID}")
print(f"   Max Tokens: {request_body['inferenceConfig']['maxTokens']}")
print(f"   Temperature: {request_body['inferenceConfig']['temperature']}")

✅ Prompt and request body prepared successfully.
   Model ID: us.amazon.nova-2-lite-v1:0
   Max Tokens: 500
   Temperature: 0.7


---
## Cell 4 — Invoke the Model (Standard Response)

This cell sends the request to Amazon Bedrock and receives the complete response at once.

> **Note:** There is no output for this cell — the response is stored in a variable for use in Cell 5.

In [4]:
# Invoke Amazon Nova 2 Lite via Bedrock Runtime
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Parse the response body
response_body = json.loads(response['body'].read())

print("✅ Model invoked successfully. Response received.")

✅ Model invoked successfully. Response received.


---
## Cell 5 — Extract and Display the Generated Email

This cell extracts and prints only the email content from the full model response.

**Response structure for Nova 2 Lite:**
```
response_body["output"]["message"]["content"][0]["text"]
```

In [5]:
# Extract the generated email text from the Nova 2 Lite response structure
generated_email = response_body["output"]["message"]["content"][0]["text"]

# Display the generated email
print("=" * 60)
print("         GENERATED EMAIL RESPONSE")
print("=" * 60)
print(generated_email)
print("=" * 60)

         GENERATED EMAIL RESPONSE
### Subject: Our Sincere Apologies and Immediate Action Plan – Order # [OrderNumber]

Dear John,

Thank you for reaching out to us and sharing your concerns—I truly appreciate your loyalty and the opportunity to make this right.

First and foremost, please accept my sincerest apologies for the frustration and inconvenience you have experienced. As a valued customer of over three years, you deserve far better service than what you received, and I understand how disappointing this must have been, especially after such a long and trusting relationship with us.

We have immediately escalated your case and are conducting a full investigation into the status of your order placed two weeks ago. Our logistics and customer support teams are now actively tracking it down, and I’ve personally requested a detailed update from our shipping partner within the next 2 business hours. You will receive this update directly from me, regardless of the outcome.

In the mea

---
## Cell 6 — Streaming Response (Real-Time Output)

This cell sends the request using streaming mode — the email is printed in small chunks as it is being generated by the model, simulating a real-time output experience.

> This demonstrates how to use `invoke_model_with_response_stream` for lower latency and real-time display.

In [15]:
streaming_response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

print("=" * 60)
print("     STREAMING RESPONSE (Real-Time Generation)")
print("=" * 60)

for event in streaming_response['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if 'contentBlockDelta' in chunk:                       # ✅ key check
        delta = chunk['contentBlockDelta']['delta']        # ✅ direct key access
        if 'text' in delta:
            print(delta['text'], end='', flush=True)

print("\n" + "=" * 60)

     STREAMING RESPONSE (Real-Time Generation)
### Subject: Our Sincere Apologies and Immediate Assistance, John

Dear John,

Thank you for reaching out to us and sharing your concerns—I truly appreciate you taking the time to do so.

First and foremost, please accept my sincerest apologies for the frustration and inconvenience you've experienced. As a valued customer who has supported us for over three years, you deserve far better service than what you received, and I completely understand why you feel so disappointed.

I’m very sorry to hear about the delay with your order and the challenging experience you had when contacting our support team. Being kept on hold for such a long time and not being proactively informed about the “lost in transit” status is unacceptable, and I take full responsibility for this falling short of our standards. 

We recognize the stress this has caused you, and we’re determined to make things right immediately.

### Here’s our plan to resolve this as qui

---
## Cell 7 — Combine All Streamed Chunks into Final Output

This cell re-runs the streaming request and assembles all the chunks into one complete email, then prints the final output as a single string.

In [12]:
streaming_response_2 = bedrock_runtime.invoke_model_with_response_stream(
    modelId=MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

all_chunks = []

for event in streaming_response_2['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if 'contentBlockDelta' in chunk:                       # ✅ key check
        delta = chunk['contentBlockDelta']['delta']
        if 'text' in delta:
            all_chunks.append(delta['text'])

complete_email = ''.join(all_chunks)
print(complete_email)

### Subject: Our Sincere Apologies and Immediate Action to Resolve Your Issue

Dear John,

Thank you for reaching out and sharing your concerns with us — I truly appreciate your loyalty and your honesty. I’m deeply sorry to hear about the frustration and disappointment you’ve experienced with your recent order. Please accept my sincerest apologies for the inconvenience this has caused you after your many years of trust in our service.

At [Company Name], we pride ourselves on providing reliable, responsive service to all our customers, and clearly, we’ve fallen short in your case. I completely understand how frustrating it must be to wait two weeks without any update, especially after being put on hold for so long. You deserve better, and I’m taking your feedback very seriously.

### Here’s what we’re doing immediately to resolve your issue:

1. **Tracking & Investigation**: I’ve personally escalated your case and contacted our logistics and shipping partners to locate your package rig

---
## Summary

In this lab, you have:

| Step | What You Did |
|------|--------------|
| Cell 1 | Installed libraries and created the Bedrock Runtime client |
| Cell 2 | Defined the customer's negative feedback email |
| Cell 3 | Built the zero-shot prompt and request body for Nova 2 Lite |
| Cell 4 | Invoked the model using the standard (synchronous) API |
| Cell 5 | Extracted and displayed the generated email response |
| Cell 6 | Used streaming to display output in real-time |
| Cell 7 | Assembled all streamed chunks into the final complete email |

---

**Model Used:** Amazon Nova 2 Lite (`us.amazon.nova-2-lite-v1:0`)  
**Region:** us-east-1 (N. Virginia) via cross-region inference  
**Status:** Active — no EOL date  

> ⚠️ **Remember:** Stop or delete your SageMaker notebook instance after completing this lab to avoid unnecessary charges.